# Spatial transcriptomics of Crohn's disease: zooming into an ulcer

Companion notebook for the TensorOmics note *Spatial transcriptomics of Crohn's disease: zooming into an ulcer*. It runs the note's code blocks in order.

**Data.** Download `xenium5k_scVI_CellCharter_batch2_reannot_SCP.h5ad` (about 9 GB) from the
[Broad Single Cell Portal, study SCP3259](https://singlecell.broadinstitute.org/single_cell/study/SCP3259)
(free, Google sign-in) and put it in `data/`. See `data/DATA_SOURCES.md`.

The note: https://tensoromics.com/notes/spatial-transcriptomics-crohns-xenium/

## Fibroblasts that stay inflamed

## Xenium spatial data

## From dots to cells

## Loading Xenium data in Python

In [ ]:
import h5py, pandas as pd

# show all columns of a wide table on one line
pd.set_option("display.width", 200, "display.max_columns", None)

# open the file; we read only the cell table (obs), the 9 GB matrix stays on disk
f = h5py.File("data/xenium5k_scVI_CellCharter_batch2_reannot_SCP.h5ad", "r")
obs = f["obs"]

In [ ]:
def column(name):
    g = obs[name]
    # numbers are stored as plain arrays: read them as is
    if isinstance(g, h5py.Dataset):
        return g[:]
    # labels are stored as codes plus a list of categories: turn codes into names
    return g["categories"].asstr()[:][g["codes"][:]]

In [ ]:
# short names for the columns we need
cols = {"section": "biosample_id", "x": "x_centroid", "y": "y_centroid",
        "area": "cell_area", "dots": "transcript_counts",
        "cell_type": "megagut_lvl1_majority_voting", "outline": "segmentation_method"}

# build the table, one column at a time
cells = pd.DataFrame({short: column(name) for short, name in cols.items()})

# short names for the three ways an outline was drawn
outline_names = {"Segmented by boundary stain (ATP1A1+CD45+E-Cadherin)": "boundary stain",
                 "Segmented by interior stain (18S)": "interior stain",
                 "Segmented by nucleus expansion of 5.0µm": "nucleus expansion"}
cells["outline"] = cells.outline.map(outline_names)

In [ ]:
# number of cells (rows) and columns
print(cells.shape)

# the first three cells
print(cells.head(3).round(1))

# how many outlines came from each method
print(cells.outline.value_counts())

## The tissue, cell by cell

In [ ]:
from math import pi

# label each cell inflamed or adjacent, from the tag in its section name
cells["piece"] = (cells.section.str.lower().str.contains("inf|affected")
                  .map({True: "inflamed", False: "adjacent"}))

# sections and cells for each kind of piece
print(cells.groupby("piece").agg(sections=("section", "nunique"), cells=("section", "size")))

# the smallest, middle and largest section, in cells
print(cells.section.value_counts().agg(["min", "median", "max"]).astype(int))

In [ ]:
# share of all cells in each cell type
print(cells.cell_type.value_counts(normalize=True).round(2))

In [ ]:
# diameter of a circle with the same area as the cell
diameter = 2 * (cells.area / pi) ** 0.5

# typical cell size and number of dots
print(f"median diameter {diameter.median():.1f} µm, median dots {cells.dots.median():.0f}")

## Zooming in, from sections to genes

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

# one color per cell type, most common type first
colors = dict(zip(cells.cell_type.value_counts().index, plt.get_cmap("tab10").colors))

def inside(df, x0, y0, w):
    # the cells whose centers fall inside a square (top-left corner x0, y0; width w; in µm)
    return df[(df.x >= x0) & (df.x < x0 + w) & (df.y >= y0) & (df.y < y0 + w)]

def tidy(ax, square=None):
    # same scale on both axes, and no tick labels
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    # zoom to a square if one is given; y grows downward, as in the image
    if square:
        x0, y0, w = square
        ax.set_xlim(x0, x0 + w)
        ax.set_ylim(y0 + w, y0)
    else:
        ax.invert_yaxis()

### Sixteen sections, an inflamed and an adjacent piece from each patient

In [ ]:
def label(section, piece):
    # patient ID, plus the specimen for the one patient with two (0060032)
    specimen = re.search(r"gi\d+", section)
    patient = section[:7] + (f" ({specimen.group()})" if section.startswith("0060032") else "")
    return f"{patient} · {piece}"

# order the sections by patient, inflamed first; draw them all at the same scale
sections = cells.groupby("section").piece.first().reset_index()
sections["label"] = [label(s, p) for s, p in zip(sections.section, sections.piece)]
sections["patient"] = sections.label.str.split(" · ").str[0]
sections = sections.sort_values(["patient", "piece"], ascending=[True, False])
span = cells.groupby("section")[["x", "y"]].agg(np.ptp).max().max()

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, (_, row) in zip(axes.flat, sections.iterrows()):
    group = cells[cells.section == row.section]
    ax.scatter(group.x, group.y, s=0.05, c=group.cell_type.map(colors), linewidths=0)
    tidy(ax, (group.x.min(), group.y.min(), span))
    ax.set_title(row.label, fontsize=9)
    # frame the section we zoom into next
    if row.section == "0043587_affected":
        for spine in ax.spines.values():
            spine.set(color="black", linewidth=2.5)
plt.show()

### One inflamed section: stroma, immune patches, and epithelium

In [ ]:
# the inflamed section we zoom into, and a 1 mm square inside it (x0, y0, width in µm),
# placed around the patch below
one = cells[cells.section == "0043587_affected"]
region = (3679.39, 606.96, 1000)

def draw_section(ax):
    # most common type first, so rarer types are drawn on top and stay visible
    for cell_type, color in colors.items():
        group = one[one.cell_type == cell_type]
        ax.scatter(group.x, group.y, s=0.3, color=color, linewidths=0, label=cell_type)
    # mark the 1 mm square we zoom into next
    ax.add_patch(Rectangle(region[:2], region[2], region[2], fill=False))
    tidy(ax)

fig, ax = plt.subplots(figsize=(6, 6))
draw_section(ax)
handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=t) for t, c in colors.items()]
ax.legend(handles=handles, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.show()

### A 1 mm square over an ulcer

In [ ]:
# the 150 µm patch with the most inflammatory fibroblasts in the section (see choose_ulcer_patch.py)
patch = (4354.39, 756.96, 150)

def draw_region(ax):
    part = inside(one, *region)
    ax.scatter(part.x, part.y, s=3, c=part.cell_type.map(colors), linewidths=0)
    # mark the 150 µm patch we zoom into next
    ax.add_patch(Rectangle(patch[:2], patch[2], patch[2], fill=False))
    tidy(ax, region)

fig, ax = plt.subplots(figsize=(6, 6))
draw_region(ax)

# a color legend for the cell types
handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=t) for t, c in colors.items()]
ax.legend(handles=handles, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.show()
# what the square holds: the cell types, and the share of cells in the ulcer niche (niche 11)
niche = pd.Series(column("spatial_cluster15"), index=cells.index)
square = inside(one, *region)
print(len(square), "cells")
print(square.cell_type.value_counts(normalize=True).round(2).to_string())
print(f"in the ulcer niche: {(niche[square.index] == '11').mean():.0%}")

### A 150 µm patch, with cells at their real size

In [ ]:
patch_cells = inside(one, *patch)

def draw_patch(ax, color_of):
    for _, cell in patch_cells.iterrows():
        # a circle at the cell's center, with the same area as its outline
        radius = (cell.area / np.pi) ** 0.5
        ax.add_patch(Circle((cell.x, cell.y), radius, color=color_of(cell), alpha=0.8))
    tidy(ax, patch)

fig, ax = plt.subplots(figsize=(6, 6))
draw_patch(ax, lambda cell: colors[cell.cell_type])

# a color legend for the cell types
handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=t) for t, c in colors.items()]
ax.legend(handles=handles, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.show()
# what the patch holds: the cell types, and the share of cells in the ulcer niche
print(len(patch_cells), "cells")
print(patch_cells.cell_type.value_counts().to_string())
print(f"in the ulcer niche: {(niche[patch_cells.index] == '11').mean():.0%}")

### The three zooms side by side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
draw_section(axes[0])
draw_region(axes[1])
draw_patch(axes[2], lambda cell: colors[cell.cell_type])
plt.show()

### Inflammatory fibroblasts in the ulcer, and their marker gene

In [ ]:
# the authors' finer labels, for the cells in the patch
fine = pd.Series(column("annot_new")[patch_cells.index], index=patch_cells.index)
is_if = fine == "M COL7A1hi inflammatory fibroblast"

# the raw counts: a sparse matrix on disk, one row per cell, one column per gene
counts = f["layers"]["counts"]
genes = list(f["var"]["_index"].asstr()[:])

def gene_counts(rows, name):
    # read only the rows we need; each row stores its nonzero genes (indices) and counts (data)
    g = genes.index(name)
    out = []
    for r in rows:
        start, end = counts["indptr"][r], counts["indptr"][r + 1]
        idx, val = counts["indices"][start:end], counts["data"][start:end]
        out.append(val[idx == g].sum())
    return pd.Series(out, index=rows)

col7a1 = gene_counts(patch_cells.index, "COL7A1")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# left: inflammatory fibroblasts in red, every other cell in gray
draw_patch(axes[0], lambda cell: "#c0392b" if is_if[cell.name] else "#d0d4d9")
axes[0].set_title(f"inflammatory fibroblasts ({is_if.sum()} of {len(is_if)} cells)", fontsize=15)

# right: COL7A1 dots per cell; gray = none, light to dark purple = 1 up to the most in the patch
top = col7a1.max()
def shade(cell):
    n = col7a1[cell.name]
    return "#d0d4d9" if n == 0 else plt.cm.Purples(0.4 + 0.6 * n / top)
draw_patch(axes[1], shade)
axes[1].set_title(f"COL7A1 dots per cell (up to {top:.0f})", fontsize=15)

# keys: red vs gray on the left, the number of COL7A1 dots on the right
def key(color, label):
    return plt.Line2D([], [], marker="o", ls="", color=color, label=label)

left = [key("#c0392b", "inflammatory fibroblast"), key("#d0d4d9", "other cell")]
axes[0].legend(handles=left, loc="upper left", bbox_to_anchor=(0, -0.02), ncol=2, frameon=False,
               fontsize=13)
right = [key("#d0d4d9", "0")]
right += [key(plt.cm.Purples(0.4 + 0.6 * n / top), str(n)) for n in range(1, int(top) + 1)]
axes[1].legend(handles=right, title="COL7A1 dots", loc="upper left", bbox_to_anchor=(0, -0.02),
               ncol=len(right), frameon=False, fontsize=13, title_fontsize=13, columnspacing=0.8)
plt.show()

# how many cells hold 0, 1, 2, ... COL7A1 dots, for inflammatory fibroblasts and for other cells
group = is_if.map({True: "inflammatory fibroblast", False: "other cell"})
table = pd.crosstab(group, col7a1.astype(int), rownames=[""], colnames=["COL7A1 dots"])
print(table.reindex(columns=range(int(top) + 1), fill_value=0))

# expression: COL7A1 dots per 1,000 dots in the cell, which corrects for how many dots the cell holds
per_1000 = 1000 * col7a1 / patch_cells.dots
print(pd.DataFrame({"total dots (median)": patch_cells.dots.groupby(group).median(),
                    "COL7A1 per 1,000 dots (mean)": per_1000.groupby(group).mean().round(1)}))

# the broad labels of the inflammatory fibroblasts
print(patch_cells.cell_type[is_if].value_counts().to_string())

In [ ]:
# inflammatory fibroblasts in every section, from most to fewest
fine_all = pd.Series(column("annot_new"), index=cells.index)
is_if_all = fine_all == "M COL7A1hi inflammatory fibroblast"
per_section = is_if_all.groupby([cells.piece, cells.section]).sum().sort_values(ascending=False)
print(per_section.to_string())
print(f"share in inflamed pieces: {is_if_all[cells.piece == 'inflamed'].sum() / is_if_all.sum():.0%}")